# Credit Score Classification: Reproducible Model Comparison

Portfolio notebook comparing an L2 logistic-regression baseline and a regularized deep neural network. **Educational use only; not credit-decision software.**

## Reproducible workflow

The threshold, imputation, and scaling are learned from training data only; validation and test data are held out from preprocessing fits.

In [ ]:
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from tensorflow import keras

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)

In [ ]:
TRAIN_PATH, TEST_PATH, TARGET = 'CreditScore_train.csv', 'CreditScore_test.csv', 'y'
train_raw, test_raw = pd.read_csv(TRAIN_PATH), pd.read_csv(TEST_PATH)
features = [column for column in train_raw.columns if column != TARGET]
if TARGET not in test_raw or set(features) != set(test_raw.columns) - {TARGET}:
    raise ValueError('Training and test files must contain matching features and a y target column.')

threshold = train_raw[TARGET].median()
X, y = train_raw[features], (train_raw[TARGET] > threshold).astype(int)
X_test, y_test = test_raw[features], (test_raw[TARGET] > threshold).astype(int)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, random_state=SEED, stratify=y)
print(f'Training-derived threshold: {threshold:.6f}; features: {len(features)}')

In [ ]:
preprocessor = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
X_train = preprocessor.fit_transform(X_train)
X_val = preprocessor.transform(X_val)
X_test = preprocessor.transform(X_test)

def evaluate(name, y_true, y_pred):
    result = {'model': name, 'accuracy': accuracy_score(y_true, y_pred), 'precision': precision_score(y_true, y_pred, zero_division=0), 'recall': recall_score(y_true, y_pred, zero_division=0), 'f1_score': f1_score(y_true, y_pred, zero_division=0)}
    print(pd.Series(result).drop('model'))
    sns.heatmap(confusion_matrix(y_true, y_pred), annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])
    plt.title(name); plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.show()
    return result

In [ ]:
logistic = LogisticRegression(penalty='l2', C=0.1, class_weight='balanced', max_iter=2000, random_state=SEED)
logistic.fit(X_train, y_train)
logistic_result = evaluate('Logistic regression — test', y_test, logistic.predict(X_test))

In [ ]:
dnn = keras.Sequential([keras.Input(shape=(X_train.shape[1],)), keras.layers.Dense(128, activation='relu', kernel_regularizer=keras.regularizers.l2(0.01)), keras.layers.BatchNormalization(), keras.layers.Dropout(0.30), keras.layers.Dense(64, activation='relu', kernel_regularizer=keras.regularizers.l2(0.01)), keras.layers.BatchNormalization(), keras.layers.Dropout(0.30), keras.layers.Dense(1, activation='sigmoid')])
dnn.compile(optimizer=keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy', keras.metrics.Precision(), keras.metrics.Recall()])
callbacks = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True), keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=3, factor=0.5)]
history = dnn.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=100, batch_size=32, callbacks=callbacks, verbose=1)
dnn_predictions = (dnn.predict(X_test, verbose=0).ravel() >= 0.50).astype(int)
dnn_result = evaluate('Regularized DNN — test', y_test, dnn_predictions)
display(pd.DataFrame([logistic_result, dnn_result]).set_index('model').style.format('{:.2%}'))
plt.plot(history.history['loss'], label='Training loss'); plt.plot(history.history['val_loss'], label='Validation loss'); plt.legend(); plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.show()

## Limitations

Credit-risk model selection requires calibration, fairness, stability, explainability, governance, and regulatory assessment in addition to performance metrics. The median threshold is an educational baseline, not a production policy. The original assignment notebook is retained at the repository root for reference.